# 02 — Embeddings & Indices

**Inputs:**
- `data/processed/chunks.parquet` — 67,599 chunks from Notebook 01

**Outputs:**
- `data/processed/embeddings.npy` — float32 (67,599 × 1,024 ≈ 274 MB), L2-normalised BGE-large vectors
- `data/indices/chroma_textbooks/` — ChromaDB persistent collection `medqa_textbooks_bge_400` (cosine HNSW)
- `data/indices/bm25.pkl` — pickled `{"index": BM25Okapi, "chunk_ids": […]}`

**Decisions applied** (from [plan.md §0](../plan.md)):
| # | Setting | Value |
|---|---|---|
| 2 | Embedding model | `BAAI/bge-large-en-v1.5` (1024-d, 335M, 512-token max) |
| 3 | Vector DB | ChromaDB persistent, cosine HNSW |
| 4 | Sparse index | rank-bm25 (Okapi) |
| 12 | Top-k passed to LLM | k=5 (used at retrieval time, not here) |

**BGE prefix convention.** Passages are embedded *without* a prefix; queries are embedded *with* `"Represent this sentence for searching relevant passages: "`. This is enforced inside `src/data/embedder.py` so callers don't have to remember.

**Resumability.** The full-embed cell is idempotent — if `embeddings.npy` already exists, it loads from disk and skips re-embedding. ChromaDB build uses `overwrite=True` so reruns produce a fresh collection. BM25 is rebuilt from scratch (it's quick, ~30 sec).

**Time budget (M1 Pro, MPS).** ≈ 22 min embed + 1 min Chroma + 0.5 min BM25 ≈ **~25 min total** end-to-end (or ~50 min if MPS isn't available).

## 1. Setup

In [2]:
import sys, os, time, textwrap, logging
from pathlib import Path

import numpy as np
import pandas as pd

# Quiet chromadb's posthog telemetry warnings (cosmetic API mismatch in newer posthog)
os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb.telemetry").setLevel(logging.ERROR)
logging.getLogger("chromadb").setLevel(logging.WARNING)

# Resolve repo root (notebook may be opened from repo root or notebooks/)
cwd = Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.embedder import (
    load_bge, embed_passages, embed_queries,
    BGE_MODEL_NAME, BGE_QUERY_PREFIX, EMBEDDING_DIM, best_device,
)
from src.data.indices import (
    build_chroma, load_chroma, build_bm25, load_bm25, bm25_top_k,
    CHROMA_COLLECTION_NAME,
)

CHUNKS_PATH     = REPO_ROOT / "data" / "processed" / "chunks.parquet"
EMBEDDINGS_PATH = REPO_ROOT / "data" / "processed" / "embeddings.npy"
CHROMA_DIR      = REPO_ROOT / "data" / "indices" / "chroma_textbooks"
BM25_PATH       = REPO_ROOT / "data" / "indices" / "bm25.pkl"
EMBEDDINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
BM25_PATH.parent.mkdir(parents=True, exist_ok=True)

device = best_device()
print(f"Repo root:    {REPO_ROOT}")
print(f"Device:       {device}")
print(f"Model:        {BGE_MODEL_NAME}")
print(f"Embed dim:    {EMBEDDING_DIM}")
print(f"Query prefix: {BGE_QUERY_PREFIX!r}")
print(f"Chroma dir:   {CHROMA_DIR}")
print(f"BM25 path:    {BM25_PATH}")


Repo root:    /Users/rajak/Workstation/Projects/myGitHub/thesis-project
Device:       mps
Model:        BAAI/bge-large-en-v1.5
Embed dim:    1024
Query prefix: 'Represent this sentence for searching relevant passages: '
Chroma dir:   /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/indices/chroma_textbooks
BM25 path:    /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/indices/bm25.pkl


## 2. Load chunks

In [3]:
chunks_df = pd.read_parquet(CHUNKS_PATH)
print(f"Loaded {len(chunks_df):,} chunks across {chunks_df.book_name.nunique()} books "
      f"(mean {chunks_df.n_tokens.mean():.0f} tokens, max {chunks_df.n_tokens.max()})")
chunks_df.head(3)


Loaded 67,599 chunks across 18 books (mean 324 tokens, max 402)


,chunk_id,book_name,text,n_tokens,n_chars
0,Anatomy_Gray_chunk_00000,Anatomy_Gray,What is anatomy?\n\nAnatomy includes those str...,357,1910
1,Anatomy_Gray_chunk_00001,Anatomy_Gray,How can gross anatomy be studied?\n\nThe term ...,328,1564
2,Anatomy_Gray_chunk_00002,Anatomy_Gray,Each of these approaches has benefits and defi...,386,1934


## 3. Load BGE-large

The model is 335 M params, ~1.3 GB on disk. **First run** downloads from HuggingFace (~2 min); subsequent runs load from `~/.cache/huggingface/` (~5 sec).

In [4]:
%time model = load_bge(device=device)
print(f"\nLoaded {BGE_MODEL_NAME} on {device}")


CPU times: user 213 ms, sys: 570 ms, total: 784 ms
Wall time: 18.7 s

Loaded BAAI/bge-large-en-v1.5 on mps


## 4. Tiny smoke test

Embed 5 chunks → verify shape `(5, 1024)`, dtype `float32`, all rows L2-normalised. ~2 seconds. If anything looks off here, fix it before launching the full corpus (which costs ~22 min).

In [5]:
sample_texts = chunks_df.text.head(5).tolist()
%time sample_embs = embed_passages(model, sample_texts, batch_size=5, show_progress=False)

print(f"\nShape:  {sample_embs.shape}")
print(f"Dtype:  {sample_embs.dtype}")
norms = np.linalg.norm(sample_embs, axis=1)
print(f"Norms:  min={norms.min():.4f}  max={norms.max():.4f}  (expected ≈ 1.0 — L2-normalised)")

assert sample_embs.shape == (5, EMBEDDING_DIM), "wrong shape"
assert sample_embs.dtype == np.float32, "wrong dtype"
assert np.allclose(norms, 1.0, atol=1e-3), "embeddings not unit-normalised"
print("\n✓ smoke checks passed")


CPU times: user 107 ms, sys: 204 ms, total: 311 ms
Wall time: 727 ms

Shape:  (5, 1024)
Dtype:  float32
Norms:  min=1.0000  max=1.0000  (expected ≈ 1.0 — L2-normalised)

✓ smoke checks passed


## 5. Pre-flight time estimate

Time **one** batch of 32 chunks → extrapolate to the full 67,599. The number printed below is what you're committing to when you run the next cell.

In [6]:
batch_size = 32
n_chunks = len(chunks_df)
n_batches = (n_chunks + batch_size - 1) // batch_size

t0 = time.time()
_ = embed_passages(model, chunks_df.text.head(batch_size).tolist(),
                    batch_size=batch_size, show_progress=False)
batch_seconds = time.time() - t0

est_minutes = batch_seconds * n_batches / 60
out_mb = n_chunks * EMBEDDING_DIM * 4 / 1e6
print(f"Device:           {device}")
print(f"Batch size:       {batch_size}")
print(f"Total batches:    {n_batches:,}")
print(f"One-batch time:   {batch_seconds:.2f} s")
print(f"Estimated total:  ~{est_minutes:.1f} min")
print(f"Output size:      ~{out_mb:.0f} MB on disk (float32, L2-normalised)")


Device:           mps
Batch size:       32
Total batches:    2,113
One-batch time:   3.36 s
Estimated total:  ~118.4 min
Output size:      ~277 MB on disk (float32, L2-normalised)


## 6. Full embed → `embeddings.npy`

Embed all 67,599 chunks. **Resumable**: if `embeddings.npy` already exists, this cell loads from disk and skips re-embedding. To force a fresh embed, delete the file first.

Estimated time: ~22 min on MPS, ~45–50 min on CPU.

In [7]:
if EMBEDDINGS_PATH.exists():
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f"Loaded existing embeddings: {EMBEDDINGS_PATH}")
    print(f"Shape: {embeddings.shape}  Dtype: {embeddings.dtype}")
else:
    print(f"Embedding {len(chunks_df):,} chunks at batch size {batch_size}...")
    t0 = time.time()
    embeddings = embed_passages(
        model, chunks_df.text.tolist(),
        batch_size=batch_size, show_progress=True,
    )
    elapsed_min = (time.time() - t0) / 60
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"\nEmbedded in {elapsed_min:.1f} min")
    print(f"Saved {EMBEDDINGS_PATH}  ({embeddings.nbytes / 1e6:.1f} MB)")

print(f"\nFinal shape: {embeddings.shape}")
print(f"Sample norm: {np.linalg.norm(embeddings[0]):.4f} (expected ≈ 1.0)")
assert embeddings.shape == (len(chunks_df), EMBEDDING_DIM), "embedding/chunks count mismatch"


Loaded existing embeddings: /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/processed/embeddings.npy
Shape: (67599, 1024)  Dtype: float32

Final shape: (67599, 1024)
Sample norm: 1.0000 (expected ≈ 1.0)


## 7. Build ChromaDB collection

Build the persistent collection `medqa_textbooks_bge_400` at `data/indices/chroma_textbooks/` with cosine HNSW. Adds in batches of 1,000. Idempotent: `overwrite=True` drops any existing collection of the same name first.

In [8]:
%time coll = build_chroma(chunks_df, embeddings, persist_dir=CHROMA_DIR, batch_size=1000, overwrite=True)
print(f"\nCollection name: {coll.name}")
print(f"Count:           {coll.count():,}")
print(f"Persist dir:     {CHROMA_DIR}")

assert coll.count() == len(chunks_df), \
    f"ChromaDB count {coll.count()} != chunks {len(chunks_df)}"
print("\n✓ ChromaDB built and counts match")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


CPU times: user 3min 16s, sys: 16.5 s, total: 3min 33s
Wall time: 1min 39s

Collection name: medqa_textbooks_bge_400
Count:           67,599
Persist dir:     /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/indices/chroma_textbooks

✓ ChromaDB built and counts match


## 8. Build BM25 index

Tokenisation: lowercase + alphanumeric word split (matches what the retriever will do at query time). Saves a pickled payload `{"index": BM25Okapi, "chunk_ids": [...]}` so a BM25 score index → `chunk_id` is an O(1) lookup.

In [9]:
%time bm25_payload = build_bm25(chunks_df, BM25_PATH)
size_mb = BM25_PATH.stat().st_size / 1e6
print(f"\nBM25 saved: {BM25_PATH}")
print(f"Size:       {size_mb:.1f} MB")
print(f"Chunk IDs:  {len(bm25_payload['chunk_ids']):,}")
assert len(bm25_payload['chunk_ids']) == len(chunks_df)
print("\n✓ BM25 built")


CPU times: user 6.79 s, sys: 828 ms, total: 7.62 s
Wall time: 8.01 s

BM25 saved: /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/indices/bm25.pkl
Size:       105.8 MB
Chunk IDs:  67,599

✓ BM25 built


## 9. Smoke query — both indices

Acceptance test from [docs/todo.md §2.2](../docs/todo.md): query *"What is the first-line treatment for community-acquired pneumonia?"* on both indices. Top-3 from each should clearly relate to pneumonia / antibiotics.

We deliberately reload both indices from disk before querying — that confirms persistence works end-to-end (the next notebook will only ever see the on-disk artefacts, never the in-memory objects).

In [10]:
SMOKE_QUERY = "What is the first-line treatment for community-acquired pneumonia?"
WIDTH = 100

# Reload both from disk (proves persistence)
chroma_coll_loaded = load_chroma(CHROMA_DIR)
bm25_loaded        = load_bm25(BM25_PATH)
print(f"Loaded ChromaDB collection (count={chroma_coll_loaded.count():,}) and BM25 from disk\n")

# --- Dense (ChromaDB / BGE-large) ---
q_emb = embed_queries(model, [SMOKE_QUERY])
chroma_res = chroma_coll_loaded.query(query_embeddings=q_emb.tolist(), n_results=3)

print("=" * WIDTH)
print(f"QUERY: {SMOKE_QUERY}")
print("=" * WIDTH)
print("\n--- Dense (ChromaDB / BGE-large) top-3 ---")
for chunk_id, doc, dist in zip(chroma_res["ids"][0], chroma_res["documents"][0], chroma_res["distances"][0]):
    print(f"\n[{chunk_id}]  cosine_distance={dist:.4f}")
    print(textwrap.fill(doc[:500], width=WIDTH))
    if len(doc) > 500:
        print("...")

# --- Sparse (BM25) ---
print("\n\n--- Sparse (BM25) top-3 ---")
for chunk_id, score in bm25_top_k(SMOKE_QUERY, bm25_loaded, k=3):
    text = chunks_df.loc[chunks_df.chunk_id == chunk_id, "text"].iloc[0]
    print(f"\n[{chunk_id}]  bm25_score={score:.4f}")
    print(textwrap.fill(text[:500], width=WIDTH))
    if len(text) > 500:
        print("...")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Loaded ChromaDB collection (count=67,599) and BM25 from disk



Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


QUERY: What is the first-line treatment for community-acquired pneumonia?

--- Dense (ChromaDB / BGE-large) top-3 ---

[InternalMed_Harrison_chunk_05120]  cosine_distance=0.2783
Management of bacteremic pneumococcal pneumonia also is controversial. Data from nonrandomized
studies suggest that combination therapy (especially macrolide/β-lactam) is associated with a lower
mortality rate than monotherapy, particularly in severely ill patients. The exact reason is unknown,
but possible explanations include an additive or synergistic antibacterial effect, antimicrobial
tolerance, atypical co-infection, or the immunomodulatory effects of the macrolides.  For CAP
patients admi
...

[InternalMed_Harrison_chunk_05116]  cosine_distance=0.2868
Since the etiology of CAP is rarely known at the outset of treatment, initial therapy is usually
empirical, designed to cover the most likely pathogens (Table 153-5). In all cases, antibiotic
treatment should be initiated as expeditiously as possible. The C

## 10. Acceptance summary

In [11]:
rows = [
    ("chunks.parquet rows",     f"{len(chunks_df):,}"),
    ("embeddings.npy shape",    f"{embeddings.shape}"),
    ("embeddings.npy size",     f"{EMBEDDINGS_PATH.stat().st_size / 1e6:.1f} MB"),
    ("ChromaDB collection",     CHROMA_COLLECTION_NAME),
    ("ChromaDB count",          f"{chroma_coll_loaded.count():,}"),
    ("ChromaDB on-disk size",   f"{sum(p.stat().st_size for p in CHROMA_DIR.rglob('*') if p.is_file()) / 1e6:.1f} MB"),
    ("BM25 size",               f"{BM25_PATH.stat().st_size / 1e6:.1f} MB"),
    ("BM25 chunk_ids",          f"{len(bm25_loaded['chunk_ids']):,}"),
]
for k, v in rows:
    print(f"  {k:<26} {v}")

assert chroma_coll_loaded.count() == len(chunks_df)
assert len(bm25_loaded['chunk_ids']) == len(chunks_df)
assert embeddings.shape == (len(chunks_df), EMBEDDING_DIM)
print("\n✓ all deliverables on disk; counts match across the three indices")


  chunks.parquet rows        67,599
  embeddings.npy shape       (67599, 1024)
  embeddings.npy size        276.9 MB
  ChromaDB collection        medqa_textbooks_bge_400
  ChromaDB count             67,599
  ChromaDB on-disk size      1124.2 MB
  BM25 size                  105.8 MB
  BM25 chunk_ids             67,599

✓ all deliverables on disk; counts match across the three indices


---

**Done.** Both indices on disk, count parity verified, smoke query returns sensible pneumonia content from both dense and sparse retrievers.

**Next:** Notebook 03 smoke-tests the end-to-end **retrieve → prompt → Groq → parse** path on 3 dev questions before any Phase 4 experiment touches the full 12,723.